In [1]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder

In [2]:
train = pd.read_parquet('merge/result/Segment_merge_ver_02.parquet')
test = pd.read_parquet('merge/result/Segment_merge_test_ver_02.parquet')

In [3]:
# 타겟과 피처 분리
X_train = train.drop(columns=['ID', 'Segment'])
y_train = train['Segment']
X_test = test.drop(columns=['ID'])

# 타겟 인코딩 (CatBoost는 숫자형 라벨을 요구함)
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)


In [4]:
# 4. 모델 정의 및 학습
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    loss_function='MultiClass',
    verbose=100,
    random_seed=42,
    task_type='GPU'  # ← GPU 환경이면 이렇게 설정 (CPU는 생략)
)

model.fit(X_train, y_train_encoded)

0:	learn: 1.3275677	total: 39.8ms	remaining: 19.8s
100:	learn: 0.3028855	total: 3.42s	remaining: 13.5s
200:	learn: 0.2924616	total: 6.78s	remaining: 10.1s
300:	learn: 0.2866372	total: 10.1s	remaining: 6.7s
400:	learn: 0.2825407	total: 13.4s	remaining: 3.32s
499:	learn: 0.2792201	total: 16.8s	remaining: 0us


In [5]:
# 예측 (레이블 숫자 → 문자 복원)
y_test_pred_encoded = model.predict(X_test).flatten()  
y_test_pred = le.inverse_transform(y_test_pred_encoded)

In [6]:
# 결과 저장
submission = pd.DataFrame({
    'ID': test['ID'],
    'Segment': y_test_pred
})
submission.to_csv('merge/result/D1_CatB_학습및예측.csv', index=False)